In [ ]:
import zipfile
import os
import glob

zip_file_path = 'character_detection.zip'
extraction_path = '.'

# 1. Unzip the dataset
print(f"Unzipping {zip_file_path}...")
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print("Unzipping complete.")
except FileNotFoundError:
    print(f"Error: {zip_file_path} not found. Please ensure the file is uploaded.")
except Exception as e:
    print(f"An error occurred during unzipping: {e}")

# Define the expected base directory after extraction
base_dir = os.path.join(extraction_path, 'character_detection')

# 2. Verify folder structure
print("\nVerifying folder structure...")
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

structure_ok = True
if not os.path.exists(base_dir):
    print(f"Error: Base directory '{base_dir}' not found.")
    structure_ok = False
if not os.path.exists(images_dir):
    print(f"Error: Images directory '{images_dir}' not found.")
    structure_ok = False
if not os.path.exists(labels_dir):
    print(f"Error: Labels directory '{labels_dir}' not found.")
    structure_ok = False

if structure_ok:
    print("Folder structure verified: 'character_detection/images' and 'character_detection/labels' exist.")

    # 3. Print number of images and labels
    print("\nCounting images and labels...")
    image_count = 0
    label_count = 0

    # Count images (assuming common image extensions)
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']:
        image_count += len(glob.glob(os.path.join(images_dir, ext)))

    # Count labels (assuming .txt files)
    label_count = len(glob.glob(os.path.join(labels_dir, '*.txt')))

    print(f"Number of images found: {image_count}")
    print(f"Number of labels found: {label_count}")
else:
    print("Folder structure verification failed. Cannot count images and labels.")


Unzipping character_detection.zip...
Unzipping complete.

Verifying folder structure...
Folder structure verified: 'character_detection/images' and 'character_detection/labels' exist.

Counting images and labels...
Number of images found: 2503
Number of labels found: 2503


First, let's upload an image to test the model.

In [ ]:
from google.colab import files
import os

# Upload one image file
uploaded = files.upload()

# Get the filename of the uploaded image
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  uploaded_image_path = fn
  break # Assuming only one file will be uploaded for this task

Saving IMG_20260402_074930.jpg to IMG_20260402_074930 (1).jpg
Saving IMG_20260402_074924.jpg to IMG_20260402_074924 (1).jpg
Saving IMG_20260402_074921.jpg to IMG_20260402_074921 (1).jpg
Saving IMG_20260402_074915.jpg to IMG_20260402_074915 (1).jpg
Saving IMG_20260402_074909.jpg to IMG_20260402_074909 (1).jpg
Saving IMG_20260402_074905.jpg to IMG_20260402_074905 (1).jpg
Saving IMG_20260402_074859.jpg to IMG_20260402_074859 (1).jpg
User uploaded file "IMG_20260402_074930 (1).jpg" with length 2732525 bytes


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image # Import the Image module

# Load the trained model using the absolute path
model = YOLO("/content/runs/detect/train/weights/best.pt")

# Perform detection on the uploaded image
# Ensure 'uploaded_image_path' is defined from the previous cell
if 'uploaded_image_path' in globals() and os.path.exists(uploaded_image_path):
    print(f"Running inference on {uploaded_image_path}...")
    results = model(uploaded_image_path)

    # Display the results
    # results[0].show() # This opens a new window, which might not be ideal in Colab

    # To display directly in Colab, we can save the annotated image and show it
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
        display(im)
        # im.save('results.jpg')  # save image
else:
    print(f"Error: The image file '{uploaded_image_path}' was not found or not uploaded.")
    print("Please ensure you run the file upload cell first and successfully upload an image.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/train/weights/best.pt'

In [ ]:
import yaml
import os

# Define paths based on the previous extraction
dataset_path = './character_detection'
data_yaml_path = os.path.abspath(os.path.join(dataset_path, 'data.yaml')) # Use absolute path for data.yaml

# Ensure the parent directory exists
parent_dir = os.path.dirname(data_yaml_path)
os.makedirs(parent_dir, exist_ok=True)

data = {
    'path': dataset_path,  # Path to the dataset root (can be relative in yaml, but resolved by YOLO)
    'train': 'images',     # Relative path to train images from 'path'
    'val': 'images',       # Relative path to val images from 'path' (as requested)
    'nc': 1,               # Number of classes
    'names': ['character'] # Class names
}

# Create the data.yaml file
print(f"Attempting to create {data_yaml_path}...")
try:
    with open(data_yaml_path, 'w') as outfile:
        yaml.dump(data, outfile, default_flow_style=False)
    print("data.yaml created successfully:")
    with open(data_yaml_path, 'r') as file:
        print(file.read())
except Exception as e:
    print(f"Error creating data.yaml: {e}")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Does parent directory '{parent_dir}' exist? {os.path.exists(parent_dir)}")
    print(f"Is parent directory '{parent_dir}' a directory? {os.path.isdir(parent_dir)}")
    print(f"Contents of current directory {os.getcwd()}: {os.listdir(os.getcwd())}")
    if os.path.exists(parent_dir):
        print(f"Contents of parent directory {parent_dir}: {os.listdir(parent_dir)}")

Attempting to create /content/character_detection/data.yaml...
data.yaml created successfully:
names:
- character
nc: 1
path: ./character_detection
train: images
val: images



In [ ]:
# Install Ultralytics
!pip install ultralytics -qq

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8n model
model = YOLO('yolov8n.pt')

# Train the model with specified parameters
# 'data.yaml' is already created in the './character_detection/' directory
# 'imgsz=640' sets the image size
# 'epochs=20' sets the number of training epochs
# 'batch=-1' tells YOLOv8 to automatically adjust the batch size for the GPU
# 'device=0' uses the first available GPU (assuming one is present)
results = model.train(data='./character_detection/data.yaml',
                      imgsz=640,
                      epochs=20,
                      batch=-1,
                      device=0) # use device=0 for GPU, or 'cpu' for CPU

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./character_detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective

In [ ]:
import os
from google.colab import files

# Define the path to the best.pt file
# Ultralytics typically saves models in runs/detect/train/weights/
best_model_path = '/content/runs/detect/train/weights/best.pt'

# 1. Check if best.pt exists and print its path
if os.path.exists(best_model_path):
    print(f"Found best.pt model at: {best_model_path}")

    # 2. Add code to download it
    print("\nDownloading best.pt to your local machine...")
    try:
        files.download(best_model_path)
        print("Download initiated. Check your browser's downloads.")
    except Exception as e:
        print(f"Error during download: {e}. If the download doesn't start, \n" \
              "you might need to manually download it or run this cell again in a Colab environment.")
else:
    print(f"Error: best.pt not found at {best_model_path}. Please ensure training completed successfully.")


In [ ]:
!pip install ultralytics -qq

from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image # Import the Image module

# Load the trained model
model = YOLO("runs/detect/train/weights/best.pt")

# Perform detection on the uploaded image
# Ensure 'uploaded_image_path' is defined from the previous cell
if 'uploaded_image_path' in globals() and os.path.exists(uploaded_image_path):
    print(f"Running inference on {uploaded_image_path}...")
    results = model(uploaded_image_path)

    # Display the results
    # results[0].show() # This opens a new window, which might not be ideal in Colab

    # To display directly in Colab, we can save the annotated image and show it
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
        display(im)
        # im.save('results.jpg')  # save image
else:
    print(f"Error: The image file '{uploaded_image_path}' was not found or not uploaded.")
    print("Please ensure you run the file upload cell first and successfully upload an image.")